# ZPDPatch Experiment Analysis

이 notebook은 논문의 모든 결과 분석과 figure 생성을 관리한다. 새 9:1 problem split에서 완료 마커와 기대 row 수를 모두 충족한 산출물만 분석한다.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "paper" / "main.tex").exists():
    raise RuntimeError("Run expr.ipynb from the repository root.")

ANALYSIS_DIR = ROOT / "outputs" / "analysis"
EXPERIMENT_ROOT = ROOT / "outputs" / "split-90-10"
DATASET_DIR = EXPERIMENT_ROOT / "datasets"
FINAL_DIR = EXPERIMENT_ROOT / "final"
MARKER_DIR = EXPERIMENT_ROOT / "markers"
FIGURE_DIR = ROOT / "paper" / "Figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_DATASETS = {
    "Seen": DATASET_DIR / "seen-test.filtered.jsonl",
    "Unseen": DATASET_DIR / "unseen-test.filtered.jsonl",
}
METHOD_FILES = {
    "Seen": {"Zero-shot": "zero-shot", "LSGen": "lsgen", "ZPDPatch": "zpdpatch"},
    "Unseen": {"Zero-shot": "zero-shot", "ZPDPatch": "zpdpatch"},
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 180,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLORS = {
    "Progress": "#2A9D8F",
    "Strict": "#E9A23B",
    "Answer": "#457B9D",
    "Sequential": "#5B5F97",
}
HATCHES = {"Progress": "", "Strict": "//", "Answer": "xx", "Sequential": ".."}

print(f"Repository: {ROOT}")
print(f"Analysis outputs: {ANALYSIS_DIR}")
print(f"Paper figures: {FIGURE_DIR}")

Repository: /Users/cdw/VSCode/zpd-apr
Analysis outputs: /Users/cdw/VSCode/zpd-apr/outputs/analysis
Paper figures: /Users/cdw/VSCode/zpd-apr/paper/Figures


## Common metrics

- PR: mean fixed-program testcase pass rate
- RR: fraction of repaired programs
- IR: fraction whose testcase pass rate improves
- ATT: problem-level online repair wall-clock time divided by the number of buggy programs; shared source/oracle validation is offline
- TED: mean AST distance on repaired and parseable programs

In [2]:
def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def count_jsonl(path: Path) -> int | None:
    if not path.exists():
        return None
    return sum(1 for line in path.open(encoding="utf-8") if line.strip())


EXPECTED = {setting: count_jsonl(path) for setting, path in EVALUATION_DATASETS.items()}
RQ15_COMPLETE = (MARKER_DIR / "EVAL-COMPLETE").is_file()
RQ34_COMPLETE = (MARKER_DIR / "RQ34-EVAL").is_file()
RQ2_COMPLETE = (MARKER_DIR / "RQ2-EVAL").is_file()


def mean_or_nan(values: list[float]) -> float:
    return float(np.mean(values)) if values else float("nan")


def problem_level_att(rows: list[dict[str, Any]]) -> float:
    timings: dict[str, tuple[float, int]] = {}
    for row in rows:
        problem_id = str(row.get("problem_id", ""))
        elapsed = row.get("problem_repair_time_sec")
        count = row.get("problem_buggy_count")
        if problem_id and elapsed is not None and count is not None:
            timings.setdefault(problem_id, (float(elapsed), int(count)))
    total_count = sum(count for _, count in timings.values())
    return (
        sum(elapsed for elapsed, _ in timings.values()) / total_count
        if total_count
        else float("nan")
    )


def wilson_interval(successes: int, total: int, z: float = 1.959963984540054) -> tuple[float, float]:
    if total == 0:
        return float("nan"), float("nan")
    proportion = successes / total
    denominator = 1.0 + z**2 / total
    center = (proportion + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt(
        proportion * (1.0 - proportion) / total + z**2 / (4 * total**2)
    ) / denominator
    return 100 * (center - margin), 100 * (center + margin)


def summarize_rows(
    rows: list[dict[str, Any]],
    *,
    att_fallback: float | None = None,
) -> dict[str, float | int]:
    repaired = [row for row in rows if bool(row.get("repaired"))]
    ted_bf = [
        float(value)
        for row in repaired
        if (value := row.get("ted_buggy_fixed", row.get("tree_edit_distance")))
        is not None
    ]
    ted_fo = [
        float(row["ted_fixed_oracle"])
        for row in repaired
        if row.get("ted_fixed_oracle") is not None
    ]
    att = problem_level_att(rows)
    if np.isnan(att) and att_fallback is not None:
        att = float(att_fallback)
    rr_low, rr_high = wilson_interval(len(repaired), len(rows))
    improved_count = sum(bool(row.get("improved")) for row in rows)
    ir_low, ir_high = wilson_interval(improved_count, len(rows))
    return {
        "N": len(rows),
        "PR": 100 * mean_or_nan([float(row.get("fixed_pass_rate", 0.0)) for row in rows]),
        "RR": 100 * mean_or_nan([float(bool(row.get("repaired"))) for row in rows]),
        "RR_CI_LOW": rr_low,
        "RR_CI_HIGH": rr_high,
        "IR": 100 * mean_or_nan([float(bool(row.get("improved"))) for row in rows]),
        "IR_CI_LOW": ir_low,
        "IR_CI_HIGH": ir_high,
        "ATT": att,
        "TED_BF": mean_or_nan(ted_bf),
        "TED_FO": mean_or_nan(ted_fo),
        "#Patch": mean_or_nan([float(len(row.get("patches", []))) for row in rows])
        if any("patches" in row for row in rows)
        else 1.0,
    }


def completed_rows(path: Path, expected: int | None) -> list[dict[str, Any]] | None:
    if expected is None or not path.exists():
        return None
    rows = read_jsonl(path)
    return rows if len(rows) == expected else None


def rounded(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in result.select_dtypes(include=["number"]).columns:
        result[column] = result[column].round(2)
    return result


def paired_rows(
    left: list[dict[str, Any]],
    right: list[dict[str, Any]],
) -> list[tuple[dict[str, Any], dict[str, Any]]]:
    left_by_id = {str(row["example_id"]): row for row in left}
    right_by_id = {str(row["example_id"]): row for row in right}
    if set(left_by_id) != set(right_by_id):
        raise ValueError("Paired evaluations do not contain identical example IDs.")
    return [(left_by_id[key], right_by_id[key]) for key in sorted(left_by_id)]


def exact_paired_binary(
    baseline: list[dict[str, Any]],
    method: list[dict[str, Any]],
    *,
    key: str,
) -> dict[str, float | int]:
    pairs = paired_rows(baseline, method)
    method_only = sum(not bool(a.get(key)) and bool(b.get(key)) for a, b in pairs)
    baseline_only = sum(bool(a.get(key)) and not bool(b.get(key)) for a, b in pairs)
    discordant = method_only + baseline_only
    p_value = (
        float(binomtest(method_only, discordant, 0.5, alternative="two-sided").pvalue)
        if discordant
        else 1.0
    )
    return {
        "method_only": method_only,
        "baseline_only": baseline_only,
        "discordant": discordant,
        "p_value": p_value,
    }


def exact_mcnemar(
    baseline: list[dict[str, Any]],
    method: list[dict[str, Any]],
) -> dict[str, float | int]:
    result = exact_paired_binary(baseline, method, key="repaired")
    return {
        "method_only_repairs": result["method_only"],
        "baseline_only_repairs": result["baseline_only"],
        "discordant": result["discordant"],
        "p_value": result["p_value"],
    }


def paired_bootstrap_mean_difference(
    baseline: list[dict[str, Any]],
    method: list[dict[str, Any]],
    *,
    value_key: str,
    repaired_only: bool = False,
    samples: int = 10_000,
    seed: int = 2027,
) -> dict[str, float | int]:
    differences = np.asarray([
        float(b[value_key]) - float(a[value_key])
        for a, b in paired_rows(baseline, method)
        if a.get(value_key) is not None
        and b.get(value_key) is not None
        and (not repaired_only or (bool(a.get("repaired")) and bool(b.get("repaired"))))
    ])
    if not len(differences):
        return {"paired_n": 0, "mean_difference": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    rng = np.random.default_rng(seed)
    estimates = np.empty(samples, dtype=float)
    for start in range(0, samples, 500):
        size = min(500, samples - start)
        indexes = rng.integers(0, len(differences), size=(size, len(differences)))
        estimates[start:start + size] = differences[indexes].mean(axis=1)
    low, high = np.quantile(estimates, [0.025, 0.975])
    return {
        "paired_n": len(differences),
        "mean_difference": float(differences.mean()),
        "ci_low": float(low),
        "ci_high": float(high),
    }


def add_holm_adjustment(frame: pd.DataFrame, column: str = "p_value") -> pd.DataFrame:
    result = frame.copy()
    output_column = "p_holm" if column == "p_value" else f"{column}_holm"
    if result.empty:
        result[output_column] = pd.Series(dtype=float)
        return result
    values = result[column].astype(float).to_numpy()
    order = np.argsort(values)
    adjusted = np.empty(len(values), dtype=float)
    running = 0.0
    for rank, index in enumerate(order):
        candidate = min(1.0, (len(values) - rank) * values[index])
        running = max(running, candidate)
        adjusted[index] = running
    result[output_column] = adjusted
    return result

## Dataset split statistics

Table 1의 통계를 canonical manifests와 summary에서 직접 계산한다.

In [3]:
split_summary = read_json(ROOT / "data" / "splits" / "summary.json")
split_roles = {
    "seen_train": ("Seen", "Train / LSGen DB"),
    "seen_valid": ("Seen", "Validation"),
    "seen_test": ("Seen", "Test"),
    "unseen_test": ("Unseen", "Test"),
}
dataset_rows = []
for split, (group, role) in split_roles.items():
    manifest = read_jsonl(ROOT / "data" / "splits" / f"{split}.jsonl")
    group_stats = split_summary[group.lower()]
    dataset_rows.append({
        "Group": group,
        "Role": role,
        "Prob.": group_stats["problems"],
        "Tests": group_stats["testcases"],
        "Traj.": len(manifest),
        "Sub.": sum(int(row["submission_count"]) for row in manifest),
        "Prefix": sum(int(row["prefix_count"]) for row in manifest),
        "Users": len({str(row["user_id"]) for row in manifest}),
    })
dataset_statistics = pd.DataFrame(dataset_rows)
dataset_statistics.to_csv(ANALYSIS_DIR / "dataset_split_statistics.csv", index=False)
display(dataset_statistics)

sft_sources = {
    "Progress": DATASET_DIR / "train-progress.jsonl",
    "Strict": DATASET_DIR / "train-strict.jsonl",
    "Answer": DATASET_DIR / "train-answer.jsonl",
}
sft_rows = []
for adapter, path in sft_sources.items():
    if path.exists():
        records = read_jsonl(path)
        sft_rows.append({
            "Adapter": adapter,
            "Examples": len(records),
            "Problems": len({str(row["problem_id"]) for row in records}),
            "Users": len({str(row["user_id"]) for row in records}),
        })
sft_statistics = pd.DataFrame(sft_rows)
sft_statistics.to_csv(ANALYSIS_DIR / "sft_source_statistics.csv", index=False)
display(sft_statistics)

,Group,Role,Prob.,Tests,Traj.,Sub.,Prefix,Users
0,Seen,Train / LSGen DB,492,9448,16952,78874,61922,4325
1,Seen,Validation,492,9448,2119,9820,7701,1406
2,Seen,Test,492,9448,2119,9718,7599,1409
3,Unseen,Test,54,640,264,1020,756,238


,Adapter,Examples,Problems,Users
0,Progress,28208,492,4325
1,Strict,23534,492,4325
2,Answer,16951,492,4325


## RQ1 and RQ5: Repair effectiveness and problem generalization

Only methods with all expected rows are included. The shared paper table uses the same metrics for Seen and Unseen.

In [4]:
rq1_rows: list[dict[str, Any]] = []
rq1_raw: dict[tuple[str, str], list[dict[str, Any]]] = {}
for setting, expected in EXPECTED.items():
    setting_dir = FINAL_DIR / setting.lower()
    for method, stem in METHOD_FILES[setting].items():
        path = setting_dir / f"{stem}.jsonl"
        rows = completed_rows(path, expected)
        if rows is None:
            continue
        rq1_raw[(setting, method)] = rows
        metrics = summarize_rows(rows)
        rq1_rows.append({"Setting": setting, "Approach": method, **metrics})

rq1 = pd.DataFrame(rq1_rows)
rq1_columns = [
    "Setting", "Approach", "N", "PR", "RR", "RR_CI_LOW", "RR_CI_HIGH",
    "IR", "IR_CI_LOW", "IR_CI_HIGH", "ATT", "TED_BF", "TED_FO",
]
rq1 = rq1.reindex(columns=rq1_columns)
rq1.to_csv(ANALYSIS_DIR / "rq1_rq5_completed.csv", index=False)
display(rounded(rq1))

rq1_tests: list[dict[str, Any]] = []
for setting in EXPECTED:
    zpd = rq1_raw.get((setting, "ZPDPatch"))
    baselines = {
        name: rq1_raw.get((setting, name))
        for name in METHOD_FILES[setting]
        if name != "ZPDPatch"
    }
    if zpd is None or any(rows is None for rows in baselines.values()):
        continue
    for baseline_name, baseline_rows in baselines.items():
        assert baseline_rows is not None
        rq1_tests.append({
            "Setting": setting,
            "Comparison": f"ZPDPatch vs {baseline_name}",
            **exact_mcnemar(baseline_rows, zpd),
            **{
                f"PR_{key}": value
                for key, value in paired_bootstrap_mean_difference(
                    baseline_rows,
                    zpd,
                    value_key="fixed_pass_rate",
                ).items()
            },
            **{
                f"TED_BF_{key}": value
                for key, value in paired_bootstrap_mean_difference(
                    baseline_rows,
                    zpd,
                    value_key="ted_buggy_fixed",
                    repaired_only=True,
                ).items()
            },
            **{
                f"TED_FO_{key}": value
                for key, value in paired_bootstrap_mean_difference(
                    baseline_rows,
                    zpd,
                    value_key="ted_fixed_oracle",
                    repaired_only=True,
                ).items()
            },
        })
rq1_significance = pd.DataFrame(rq1_tests)
if not rq1_significance.empty:
    rq1_significance = pd.concat([
        add_holm_adjustment(group)
        for _, group in rq1_significance.groupby("Setting", sort=False)
    ], ignore_index=True)
rq1_significance.to_csv(ANALYSIS_DIR / "rq1_rq5_paired_tests.csv", index=False)
if not rq1_significance.empty:
    display(rounded(rq1_significance))

,Setting,Approach,N,PR,RR,RR_CI_LOW,RR_CI_HIGH,IR,IR_CI_LOW,IR_CI_HIGH,ATT,TED_BF,TED_FO
0,Seen,Zero-shot,1702,70.93,19.21,17.41,21.15,32.31,30.14,34.57,13.26,30.10,30.99
1,Seen,LSGen,1702,83.04,72.39,70.21,74.46,75.79,73.70,77.77,19.46,111.30,106.90
2,Seen,ZPDPatch,1702,75.11,28.61,26.52,30.81,40.78,38.46,43.13,15.01,15.97,18.16


,Setting,Comparison,method_only_repairs,baseline_only_repairs,discordant,p_value,PR_paired_n,PR_mean_difference,PR_ci_low,PR_ci_high,TED_BF_paired_n,TED_BF_mean_difference,TED_BF_ci_low,TED_BF_ci_high,TED_FO_paired_n,TED_FO_mean_difference,TED_FO_ci_low,TED_FO_ci_high,p_holm
0,Seen,ZPDPatch vs Zero-shot,230,70,300,0.0,1702,0.04,0.03,0.05,238,-9.63,-14.68,-5.50,257,-9.12,-13.77,-5.40,0.0
1,Seen,ZPDPatch vs LSGen,66,811,877,0.0,1702,-0.08,-0.10,-0.06,402,-52.13,-58.97,-45.58,421,-48.62,-55.26,-42.31,0.0


## Diagnostic analysis: Why is ZPDPatch RR limited?

This analysis separates limitations in the training objective, adapter coverage, candidate generation, and final selection. It uses only the completed Seen evaluation so that every diagnostic is computed over the same 1,702 buggy programs.

In [5]:
import ast
from collections import Counter

zpd_seen = rq1_raw.get(("Seen", "ZPDPatch"))
if zpd_seen is None:
    zpd_seen = completed_rows(
        FINAL_DIR / "seen" / "zpdpatch.jsonl",
        EXPECTED["Seen"],
    )
if zpd_seen is None:
    raise RuntimeError("The complete Seen ZPDPatch evaluation is required.")

seen_records = {
    str(row["example_id"]): row
    for row in read_jsonl(EVALUATION_DATASETS["Seen"])
}
if set(seen_records) != {str(row["example_id"]) for row in zpd_seen}:
    raise ValueError("ZPDPatch results and Seen evaluation data do not match.")

# Training objective and coverage diagnostics.
training_rows = {
    stage: read_jsonl(DATASET_DIR / f"train-{stage}.jsonl")
    for stage in ("progress", "strict", "answer")
}
training_ids = {
    stage: {str(row["example_id"]) for row in rows}
    for stage, rows in training_rows.items()
}
training_diagnostics = []
for stage, rows in training_rows.items():
    summary = read_json(EXPERIMENT_ROOT / "training-summaries" / f"{stage}.json")
    accepted_targets = sum(str(row.get("target_verdict")) == "Accepted" for row in rows)
    examples_seen_equivalent = int(summary["max_steps"]) * int(summary["effective_batch_size"])
    training_diagnostics.append({
        "Stage": stage.title(),
        "Source examples": len(rows),
        "Encoded examples": int(summary["encoded_examples"]),
        "AC targets": accepted_targets,
        "Non-AC targets": len(rows) - accepted_targets,
        "AC target rate (%)": 100 * accepted_targets / len(rows),
        "Max steps": int(summary["max_steps"]),
        "Effective batch": int(summary["effective_batch_size"]),
        "Example exposures": examples_seen_equivalent,
        "Approx. dataset coverage (%)": min(
            100.0,
            100 * examples_seen_equivalent / int(summary["encoded_examples"]),
        ),
    })
training_diagnostics = pd.DataFrame(training_diagnostics)
training_diagnostics.to_csv(
    ANALYSIS_DIR / "zpdpatch_training_diagnostics.csv",
    index=False,
)

training_overlap_rows = []
stages = ("progress", "strict", "answer")
for left_index, left in enumerate(stages):
    for right in stages[left_index + 1:]:
        intersection = training_ids[left] & training_ids[right]
        union = training_ids[left] | training_ids[right]
        training_overlap_rows.append({
            "Pair": f"{left.title()}--{right.title()}",
            "Shared examples": len(intersection),
            "Share of smaller set (%)": 100 * len(intersection) / min(
                len(training_ids[left]), len(training_ids[right])
            ),
            "Jaccard (%)": 100 * len(intersection) / len(union),
        })
training_overlap = pd.DataFrame(training_overlap_rows)
training_overlap.to_csv(
    ANALYSIS_DIR / "zpdpatch_training_overlap.csv",
    index=False,
)

# Candidate-level quality diagnostics.
candidate_rows = []
duplicate_rows = []
for result in zpd_seen:
    example_id = str(result["example_id"])
    record = seen_records[example_id]
    current_code = str(record["history"][-1]["code"])
    current_pass_rate = float(result["buggy_pass_rate"])
    previous_codes: set[str] = set()
    for patch in result.get("patches", []):
        source = str(patch["source"])
        generated_code = str(patch.get("generated_code", ""))
        normalized_text = generated_code.strip()
        fixed_pass_rate = float(patch["fixed_pass_rate"])
        delta = fixed_pass_rate - current_pass_rate
        try:
            ast.parse(generated_code)
            parseable = True
        except (SyntaxError, ValueError, TypeError):
            parseable = False
        candidate_rows.append({
            "example_id": example_id,
            "Stage": source.title(),
            "Buggy verdict": str(result.get("buggy_verdict", "")),
            "Current PR": current_pass_rate,
            "Candidate PR": fixed_pass_rate,
            "PR delta": delta,
            "AC": fixed_pass_rate == 1.0,
            "Improved": delta > 1e-12,
            "Equal": abs(delta) <= 1e-12,
            "Regressed": delta < -1e-12,
            "Unchanged code": normalized_text == current_code.strip(),
            "Duplicate of earlier stage": normalized_text in previous_codes,
            "Parseable": parseable,
            "Verdict": str(patch.get("fixed_verdict", "")),
        })
        previous_codes.add(normalized_text)

candidate_frame = pd.DataFrame(candidate_rows)
stage_diagnostics = (
    candidate_frame.groupby("Stage", sort=False)
    .agg(
        Generated=("example_id", "size"),
        **{
            "AC rate (%)": ("AC", lambda values: 100 * values.mean()),
            "Improvement rate (%)": ("Improved", lambda values: 100 * values.mean()),
            "Equal-PR rate (%)": ("Equal", lambda values: 100 * values.mean()),
            "Regression rate (%)": ("Regressed", lambda values: 100 * values.mean()),
            "Unchanged-code rate (%)": ("Unchanged code", lambda values: 100 * values.mean()),
            "Earlier-stage duplicate rate (%)": (
                "Duplicate of earlier stage",
                lambda values: 100 * values.mean(),
            ),
            "Parse failure rate (%)": ("Parseable", lambda values: 100 * (1 - values.mean())),
            "Mean PR delta (pp)": ("PR delta", lambda values: 100 * values.mean()),
        },
    )
    .reset_index()
)
stage_order = {"Progress": 0, "Strict": 1, "Answer": 2}
stage_diagnostics = stage_diagnostics.sort_values(
    "Stage", key=lambda values: values.map(stage_order)
)
stage_diagnostics.to_csv(
    ANALYSIS_DIR / "zpdpatch_stage_diagnostics.csv",
    index=False,
)
candidate_verdicts = pd.crosstab(
    candidate_frame["Stage"],
    candidate_frame["Verdict"],
).reindex(["Progress", "Strict", "Answer"]).fillna(0).astype(int)
candidate_verdicts.to_csv(ANALYSIS_DIR / "zpdpatch_candidate_verdicts.csv")
candidate_transition_matrix = pd.crosstab(
    [candidate_frame["Stage"], candidate_frame["Buggy verdict"]],
    candidate_frame["Verdict"],
).reindex(["Progress", "Strict", "Answer"], level="Stage").fillna(0).astype(int)
candidate_transition_matrix.to_csv(
    ANALYSIS_DIR / "zpdpatch_candidate_transition_matrix.csv"
)

# Exclusive failure decomposition and selector behavior.
failure_causes = Counter()
selection_sources = Counter()
case_rows = []
for result in zpd_seen:
    record = seen_records[str(result["example_id"])]
    patches = result.get("patches", [])
    current_pass_rate = float(result["buggy_pass_rate"])
    max_candidate_pass_rate = max(float(patch["fixed_pass_rate"]) for patch in patches)
    any_changed_ac = any(
        float(patch["fixed_pass_rate"]) == 1.0
        and str(patch.get("generated_code", "")).strip()
        != str(record["history"][-1]["code"]).strip()
        for patch in patches
    )
    if bool(result["repaired"]):
        cause = "Repaired"
    elif any_changed_ac:
        cause = "Changed AC candidate not selected"
    elif max_candidate_pass_rate >= 1.0:
        cause = "AC execution without code change"
    elif max_candidate_pass_rate > current_pass_rate + 1e-12:
        cause = "Partial improvement only"
    else:
        cause = "No candidate improves PR"
    failure_causes[cause] += 1
    selection_sources[str(result["selected_source"])] += 1
    case_rows.append({
        "example_id": str(result["example_id"]),
        "Buggy verdict": str(result["buggy_verdict"]),
        "Current PR": current_pass_rate,
        "History length": len(record["history"]),
        "Repaired": bool(result["repaired"]),
        "Improved": bool(result["improved"]),
        "Failure category": cause,
        "Selected source": str(result["selected_source"]),
    })

failure_frame = pd.DataFrame([
    {"Outcome": cause, "N": count, "Rate (%)": 100 * count / len(zpd_seen)}
    for cause, count in failure_causes.items()
]).sort_values("N", ascending=False)
failure_frame.to_csv(
    ANALYSIS_DIR / "zpdpatch_failure_decomposition.csv",
    index=False,
)
selection_frame = pd.DataFrame([
    {"Selected source": source, "N": count, "Rate (%)": 100 * count / len(zpd_seen)}
    for source, count in selection_sources.items()
]).sort_values("N", ascending=False)
selection_frame.to_csv(
    ANALYSIS_DIR / "zpdpatch_selection_sources.csv",
    index=False,
)

# Difficulty slices.
case_frame = pd.DataFrame(case_rows)
case_frame["Current-PR bucket"] = pd.cut(
    case_frame["Current PR"],
    bins=[-1e-12, 0.0, 0.25, 0.50, 0.75, 1.0000001],
    labels=["0", "(0,.25]", "(.25,.50]", "(.50,.75]", "(.75,1)"],
    include_lowest=True,
)
case_frame["History bucket"] = pd.cut(
    case_frame["History length"],
    bins=[0, 1, 2, 3, 4, 5, np.inf],
    labels=["1", "2", "3", "4", "5", "6+"],
)
def repair_slice(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    return (
        frame.groupby(column, observed=False)
        .agg(
            N=("example_id", "size"),
            **{
                "RR (%)": ("Repaired", lambda values: 100 * values.mean()),
                "IR (%)": ("Improved", lambda values: 100 * values.mean()),
            },
        )
        .reset_index()
    )

rr_by_pass = repair_slice(case_frame, "Current-PR bucket")
rr_by_history = repair_slice(case_frame, "History bucket")
rr_by_verdict = repair_slice(case_frame, "Buggy verdict").sort_values("N", ascending=False)
rr_by_pass.to_csv(ANALYSIS_DIR / "zpdpatch_rr_by_current_pass.csv", index=False)
rr_by_history.to_csv(ANALYSIS_DIR / "zpdpatch_rr_by_history_length.csv", index=False)
rr_by_verdict.to_csv(ANALYSIS_DIR / "zpdpatch_rr_by_buggy_verdict.csv", index=False)

display(rounded(training_diagnostics))
display(rounded(training_overlap))
display(rounded(stage_diagnostics))
display(candidate_verdicts)
display(candidate_transition_matrix)
display(rounded(failure_frame))
display(rounded(selection_frame))
display(rounded(rr_by_pass))
display(rounded(rr_by_history))
display(rounded(rr_by_verdict))

,Stage,Source examples,Encoded examples,AC targets,Non-AC targets,AC target rate (%),Max steps,Effective batch,Example exposures,Approx. dataset coverage (%)
0,Progress,28208,27573,16951,11257,60.09,400,16,6400,23.21
1,Strict,23534,23378,16951,6583,72.03,400,16,6400,27.38
2,Answer,16951,16585,16951,0,100.00,400,16,6400,38.59


,Pair,Shared examples,Share of smaller set (%),Jaccard (%)
0,Progress--Strict,23534,100.0,83.43
1,Progress--Answer,16951,100.0,60.09
2,Strict--Answer,16951,100.0,72.03


,Stage,Generated,AC rate (%),Improvement rate (%),Equal-PR rate (%),Regression rate (%),Unchanged-code rate (%),Earlier-stage duplicate rate (%),Parse failure rate (%),Mean PR delta (pp)
0,Progress,1702,21.68,31.61,29.67,38.72,23.09,0.00,0.12,2.85
1,Strict,1333,4.88,16.80,35.71,47.49,27.46,49.81,0.45,-10.91
2,Answer,1268,4.34,16.48,35.49,48.03,27.60,57.65,0.47,-10.75


Verdict,AC,CE,RE,TLE,WA
Stage,,,,,
Progress,369,3,84,667,579
Strict,65,7,81,656,524
Answer,55,7,83,641,482


Verdict                  AC  CE  RE  TLE   WA
Stage    Buggy verdict                       
Progress CE              23   3   0    8    3
         RE              69   0  69  102   22
         TLE             35   0   5  214   31
         WA             242   0  10  343  523
Strict   CE               0   4   0    7    3
         RE              12   0  63   98   20
         TLE              6   0   5  210   29
         WA              47   3  13  341  472
Answer   CE               1   5   0    6    2
         RE               5   0  68   96   12
         TLE             10   0   4  208   22
         WA              39   2  11  331  446

,Outcome,N,Rate (%)
1,No candidate improves PR,1008,59.22
0,Repaired,487,28.61
2,Partial improvement only,205,12.04
3,AC execution without code change,2,0.12


,Selected source,N,Rate (%)
1,current-fallback,1008,59.22
0,progress,474,27.85
3,strict,120,7.05
2,answer,100,5.88


,Current-PR bucket,N,RR (%),IR (%)
0,0,224,61.61,90.18
1,"(0,.25]",151,25.17,52.98
2,"(.25,.50]",314,23.89,39.81
3,"(.50,.75]",417,23.98,31.41
4,"(.75,1)",596,22.82,26.17


,History bucket,N,RR (%),IR (%)
0,1,0,NaN,NaN
1,2,802,33.04,45.51
2,3,377,30.77,43.77
3,4,208,22.12,30.77
4,5,119,23.53,40.34
5,6+,196,16.33,26.53


,Buggy verdict,N,RR (%),IR (%)
3,WA,1118,29.34,38.46
2,TLE,285,17.54,34.04
1,RE,262,32.82,50.76
0,CE,37,62.16,91.89


## Improvement study: training coverage and edit-weighted loss

The Answer adapter is isolated on a fixed validation monitor set to compare edit weights 1 and 4 at the same 400-step budget. The selected weight is then used for full-coverage training of the final enriched three-adapter cascade.

In [6]:
IMPROVEMENT_V1 = EXPERIMENT_ROOT / "improvement-v1"
MONITOR_DATASET = IMPROVEMENT_V1 / "datasets" / "seen-valid-monitor-256.jsonl"
monitor_expected = count_jsonl(MONITOR_DATASET)
monitor_records = {
    str(row["example_id"]): row
    for row in read_jsonl(MONITOR_DATASET)
} if monitor_expected else {}
answer_variants = {
    "400-step, weight 1": "answer-400-w1-monitor",
    "400-step, weight 4": "answer-400-w4-monitor",
}
answer_ablation_rows = []
for variant, stem in answer_variants.items():
    rows = completed_rows(
        IMPROVEMENT_V1 / "valid" / f"{stem}.jsonl",
        monitor_expected,
    )
    if rows is None:
        continue
    no_op = 0
    regressed = 0
    for row in rows:
        record = monitor_records[str(row["example_id"])]
        candidate = row.get("patches", [row])[0]
        if str(candidate["generated_code"]).strip() == str(record["history"][-1]["code"]).strip():
            no_op += 1
        if float(candidate["fixed_pass_rate"]) < float(row["buggy_pass_rate"]):
            regressed += 1
    answer_ablation_rows.append({
        "Variant": variant,
        **summarize_rows(rows),
        "No-op (%)": 100 * no_op / len(rows),
        "Regression (%)": 100 * regressed / len(rows),
    })
answer_ablation = pd.DataFrame(answer_ablation_rows)
answer_ablation.to_csv(
    ANALYSIS_DIR / "answer_training_ablation.csv",
    index=False,
)
if not answer_ablation.empty:
    display(rounded(answer_ablation))

,Variant,N,PR,RR,RR_CI_LOW,RR_CI_HIGH,IR,IR_CI_LOW,IR_CI_HIGH,ATT,TED_BF,TED_FO,#Patch,No-op (%),Regression (%)
0,"400-step, weight 1",256,76.01,34.38,28.83,40.39,44.92,38.95,51.05,4.84,15.57,23.56,1.0,17.19,24.61
1,"400-step, weight 4",256,74.97,33.98,28.46,39.99,47.27,41.24,53.38,4.11,30.76,34.44,1.0,0.39,34.77


## Improvement study: enriched trajectory context

This diagnostic checks execution-evidence coverage and prompt length after adding testcase feedback and AST edit summaries to the three disjoint adapter datasets.

In [7]:
from transformers import AutoTokenizer

from src.repair.prompts import build_messages, render_generation_prompt

disjoint_dir = IMPROVEMENT_V1 / "datasets"
length_tokenizer = AutoTokenizer.from_pretrained(
    ROOT / "checkpoints" / "prompt-b",
    use_fast=True,
)
context_rows = []
for stage in ("progress", "strict", "answer"):
    path = disjoint_dir / f"train-{stage}-disjoint.jsonl"
    records = read_jsonl(path)
    lengths = []
    observed_history = 0
    total_history = 0
    for record in records:
        prompt = render_generation_prompt(
            length_tokenizer,
            build_messages(record, "D", max_history=6),
        )
        prompt_ids = length_tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = length_tokenizer(
            str(record["target_code"]),
            add_special_tokens=False,
        )["input_ids"]
        lengths.append(len(prompt_ids) + len(target_ids) + 1)
        total_history += len(record["history"])
        observed_history += sum("pass_rate" in item for item in record["history"])
    values = np.asarray(lengths)
    context_rows.append({
        "Stage": stage.title(),
        "Examples": len(records),
        "Observed history (%)": 100 * observed_history / total_history,
        "Median tokens": np.median(values),
        "P95 tokens": np.quantile(values, 0.95),
        "Over 4096 (%)": 100 * np.mean(values > 4096),
        "Over 8192 (%)": 100 * np.mean(values > 8192),
    })
context_diagnostics = pd.DataFrame(context_rows)
context_diagnostics.to_csv(
    ANALYSIS_DIR / "enriched_context_diagnostics.csv",
    index=False,
)
display(rounded(context_diagnostics))

/Users/cdw/VSCode/zpd-apr/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal


Token indices sequence length is longer than the specified maximum sequence length for this model (361819 > 32768). Running this sequence through the model will result in indexing errors


<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2:

<unknown>:22: SyntaxWarning: invalid escape sequence '\j'


...

<unknown>:22: SyntaxWarning: invalid escape sequence '\j'
<unknown>:22: SyntaxWarning: invalid escape sequence '\j'
<unknown>:22: SyntaxWarning: invalid escape sequence '\j'


<unknown>:8: SyntaxWarning: invalid escape sequence '\d'


<unknown>:4: SyntaxWarning: invalid decimal literal
<unknown>:13: SyntaxWarning: invalid decimal literal
<unknown>:5: SyntaxWarning: invalid decimal literal
<unknown>:14: SyntaxWarning: invalid decimal literal
<unknown>:24: SyntaxWarning: invalid decimal literal
<unknown>:33: SyntaxWarning: invalid decimal literal
<unknown>:43: SyntaxWarning: invalid decimal literal
<unknown>:52: SyntaxWarning: invalid decimal literal
<unknown>:62: SyntaxWarning: invalid decimal literal
<unknown>:71: SyntaxWarning: invalid decimal literal
<unknown>:5: SyntaxWarning: invalid decimal literal
<unknown>:14: SyntaxWarning: invalid decimal literal
<unknown>:24: SyntaxWarning: invalid decimal literal
<unknown>:33: SyntaxWarning: invalid decimal literal
<unknown>:43: SyntaxWarning: invalid decimal literal
<unknown>:52: SyntaxWarning: invalid decimal literal
<unknown>:62: SyntaxWarning: invalid decimal literal
<unknown>:71: SyntaxWarning: invalid decimal literal
<unknown>:5: SyntaxWarning: invalid decimal liter

<unknown>:19: SyntaxWarning: invalid decimal literal
<unknown>:22: SyntaxWarning: invalid decimal literal


,Stage,Examples,Observed history (%),Median tokens,P95 tokens,Over 4096 (%),Over 8192 (%)
0,Progress,4674,82.98,1554.5,3609.7,2.91,0.17
1,Strict,6583,21.20,1594.0,3980.8,4.59,0.35
2,Answer,16951,54.10,1804.0,3833.0,3.73,0.19


## Improvement study: Answer training coverage and edit-weighted loss

A fixed, problem-diverse 256-example validation probe compares edit weights at the same 400-step budget. Every variant uses the same prompt, target data, base model, and evaluation examples; the selected weight is carried into the final full-coverage training.

In [8]:
ANSWER_ABLATION_DIR = EXPERIMENT_ROOT / "improvement-v1"
ANSWER_PROBE = ANSWER_ABLATION_DIR / "datasets" / "seen-valid-monitor-256.jsonl"
answer_expected = count_jsonl(ANSWER_PROBE)
answer_variants = {
    "400 steps, weight 1": "answer-400-w1-monitor",
    "400 steps, weight 4": "answer-400-w4-monitor",
}
answer_ablation_rows = []
for label, stem in answer_variants.items():
    path = ANSWER_ABLATION_DIR / "valid" / f"{stem}.jsonl"
    rows = completed_rows(path, answer_expected)
    if rows is None:
        continue
    answer_ablation_rows.append({"Configuration": label, **summarize_rows(rows)})
answer_ablation = pd.DataFrame(answer_ablation_rows)
if not answer_ablation.empty:
    answer_ablation = answer_ablation.reindex(columns=[
        "Configuration", "N", "PR", "RR", "IR", "ATT", "TED_BF", "TED_FO"
    ])
answer_ablation.to_csv(
    ANALYSIS_DIR / "answer_training_ablation_core.csv",
    index=False,
)
display(rounded(answer_ablation))

,Configuration,N,PR,RR,IR,ATT,TED_BF,TED_FO
0,"400 steps, weight 1",256,76.01,34.38,44.92,4.84,15.57,23.56
1,"400 steps, weight 4",256,74.97,33.98,47.27,4.11,30.76,34.44


## RQ2: Trajectory conditioning

The final table compares identically trained Full-Trajectory and Current-Code-Only variants on productive all-prefix samples. No result is inserted until both variants complete the same evaluation set.

In [9]:
rq2_files = {
    "Current Code Only": "current-code-only",
    "Full Trajectory": "full-trajectory",
}
rq2_raw: dict[str, list[dict[str, Any]]] = {name: [] for name in rq2_files}
rq2_ready = RQ2_COMPLETE
for setting in ("seen", "unseen"):
    dataset_path = EXPERIMENT_ROOT / "rq2" / "datasets" / f"{setting}-productive-allprefix.jsonl"
    if not dataset_path.exists():
        rq2_ready = False
        break
    expected = sum(1 for line in dataset_path.open(encoding="utf-8") if line.strip())
    for configuration, stem in rq2_files.items():
        path = EXPERIMENT_ROOT / "rq2" / "final" / setting / f"{stem}.jsonl"
        rows = completed_rows(path, expected)
        if rows is None:
            rq2_ready = False
            break
        rq2_raw[configuration].extend(rows)
    if not rq2_ready:
        break

if rq2_ready:
    rq2_rows = []
    for configuration, rows in rq2_raw.items():
        metrics = summarize_rows(rows)
        ted_fn = mean_or_nan([
            float(row["ted_fixed_oracle"])
            for row in rows
            if row.get("ted_fixed_oracle") is not None
        ])
        rq2_rows.append({
            "Configuration": configuration,
            "N": len(rows),
            "PR": metrics["PR"],
            "RR": metrics["RR"],
            "IR": metrics["IR"],
            "TED_FN": ted_fn,
        })
    rq2 = pd.DataFrame(rq2_rows)
    current_rows = rq2_raw["Current Code Only"]
    full_rows = rq2_raw["Full Trajectory"]
    rq2_test = pd.DataFrame([{
        "Comparison": "Full Trajectory vs Current Code Only",
        **exact_mcnemar(current_rows, full_rows),
        **{
            f"IR_{key}": value
            for key, value in exact_paired_binary(
                current_rows,
                full_rows,
                key="improved",
            ).items()
        },
        **{
            f"PR_{key}": value
            for key, value in paired_bootstrap_mean_difference(
                current_rows,
                full_rows,
                value_key="fixed_pass_rate",
            ).items()
        },
        **{
            f"TED_FN_{key}": value
            for key, value in paired_bootstrap_mean_difference(
                current_rows,
                full_rows,
                value_key="ted_fixed_oracle",
            ).items()
        },
    }])
    rq2_test.to_csv(ANALYSIS_DIR / "rq2_paired_tests.csv", index=False)
else:
    rq2 = pd.DataFrame([
        {"Configuration": "Current Code Only", "N": np.nan, "PR": np.nan, "RR": np.nan, "IR": np.nan, "TED_FN": np.nan},
        {"Configuration": "Full Trajectory", "N": np.nan, "PR": np.nan, "RR": np.nan, "IR": np.nan, "TED_FN": np.nan},
    ])

rq2.to_csv(ANALYSIS_DIR / ("rq2_final.csv" if rq2_ready else "rq2_table_layout.csv"), index=False)
display(rounded(rq2) if rq2_ready else rq2.fillna("--"))

,Configuration,N,PR,RR,IR,TED_FN
0,Current Code Only,--,--,--,--,--
1,Full Trajectory,--,--,--,--,--


## RQ3: Adapter specialization

Progress, Strict, Answer가 모두 새 Seen Test의 기대 row 수를 충족할 때만 표와 figure를 생성한다.

In [10]:
FINAL_ABLATION_DIR = EXPERIMENT_ROOT / "ablation" / "seen"
rq3_raw: dict[str, list[dict[str, Any]]] = {
    adapter: [] for adapter in ("Progress", "Strict", "Answer")
}
rq3_final = RQ34_COMPLETE and EXPECTED["Seen"] is not None
for adapter in rq3_raw:
    path = FINAL_ABLATION_DIR / f"{adapter.lower()}.jsonl"
    rows = completed_rows(path, EXPECTED["Seen"])
    if rows is None:
        rq3_final = False
        break
    rq3_raw[adapter] = rows

if not rq3_final:
    rq3_raw = {adapter: [] for adapter in rq3_raw}

rq3_rows: list[dict[str, Any]] = []
for adapter, rows in rq3_raw.items():
    metrics = summarize_rows(rows) if rows else {
        "N": np.nan, "PR": np.nan, "RR": np.nan, "IR": np.nan, "TED_BF": np.nan
    }
    rq3_rows.append({"Adapter": adapter, **metrics})

rq3 = pd.DataFrame(rq3_rows)[["Adapter", "N", "PR", "RR", "IR", "TED_BF"]]
rq3_output_stem = "rq3_adapter_specialization_final" if rq3_final else "rq3_table_layout"
rq3.to_csv(ANALYSIS_DIR / f"{rq3_output_stem}.csv", index=False)
display(rounded(rq3) if rq3_final else rq3.fillna("--"))

if rq3_final:
    rq3_test_rows = []
    for left_name, right_name in (("Progress", "Strict"), ("Progress", "Answer"), ("Strict", "Answer")):
        left_rows = rq3_raw[left_name]
        right_rows = rq3_raw[right_name]
        rq3_test_rows.append({
            "Comparison": f"{right_name} vs {left_name}",
            **exact_mcnemar(left_rows, right_rows),
            **{
                f"IR_{key}": value
                for key, value in exact_paired_binary(
                    left_rows,
                    right_rows,
                    key="improved",
                ).items()
            },
            **{
                f"PR_{key}": value
                for key, value in paired_bootstrap_mean_difference(
                    left_rows,
                    right_rows,
                    value_key="fixed_pass_rate",
                ).items()
            },
            **{
                f"TED_{key}": value
                for key, value in paired_bootstrap_mean_difference(
                    left_rows,
                    right_rows,
                    value_key="ted_buggy_fixed",
                    repaired_only=True,
                ).items()
            },
        })
    rq3_tests = add_holm_adjustment(pd.DataFrame(rq3_test_rows))
    rq3_tests = add_holm_adjustment(rq3_tests, column="IR_p_value")
    rq3_tests.to_csv(ANALYSIS_DIR / "rq3_adapter_paired_tests.csv", index=False)
    display(rounded(rq3_tests))

if rq3_final:
    fig, axes = plt.subplots(1, 2, figsize=(6.9, 2.25), gridspec_kw={"width_ratios": [1.65, 1.0]})
    x = np.arange(len(rq3))
    width = 0.22
    metric_colors = {"PR": "#457B9D", "RR": "#2A9D8F", "IR": "#E9A23B"}
    for offset, metric in enumerate(["PR", "RR", "IR"]):
        axes[0].bar(x + (offset - 1) * width, rq3[metric], width, label=metric, color=metric_colors[metric], edgecolor="#222222", linewidth=0.45)
    axes[0].set_xticks(x, rq3["Adapter"])
    axes[0].set_ylabel("Rate (%)")
    axes[0].set_ylim(0, 100)
    axes[0].grid(axis="y", color="#D9D9D9", linewidth=0.55)
    axes[0].legend(frameon=False, ncol=3, loc="upper center")
    axes[0].text(-0.12, 1.03, "(a)", transform=axes[0].transAxes, fontweight="bold")
    ted_bars = axes[1].bar(x, rq3["TED_BF"], color=[COLORS[name] for name in rq3["Adapter"]], edgecolor="#222222", linewidth=0.45)
    for bar, hatch in zip(ted_bars, [HATCHES[name] for name in rq3["Adapter"]], strict=True):
        bar.set_hatch(hatch)
    axes[1].set_xticks(x, rq3["Adapter"])
    axes[1].set_ylabel(r"$\mathrm{TED}_{B\rightarrow F}$")
    axes[1].grid(axis="y", color="#D9D9D9", linewidth=0.55)
    axes[1].text(-0.18, 1.03, "(b)", transform=axes[1].transAxes, fontweight="bold")
    fig.tight_layout(w_pad=1.25)
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURE_DIR / f"rq3_adapter_specialization.{suffix}", dpi=300)
    plt.show()

,Adapter,N,PR,RR,IR,TED_BF
0,Progress,--,--,--,--,--
1,Strict,--,--,--,--,--
2,Answer,--,--,--,--,--


## RQ4: Sequential escalation

세 single-adapter 결과와 동일한 Seen Test의 Sequential 결과가 모두 완전할 때만 비교한다.

In [11]:
rq4_sources = {
    "Progress": FINAL_ABLATION_DIR / "progress.jsonl",
    "Strict": FINAL_ABLATION_DIR / "strict.jsonl",
    "Answer": FINAL_ABLATION_DIR / "answer.jsonl",
    "Sequential": FINAL_DIR / "seen" / "zpdpatch.jsonl",
}
rq4_raw: dict[str, list[dict[str, Any]]] = {}
rq4_final = RQ34_COMPLETE and EXPECTED["Seen"] is not None
for configuration, path in rq4_sources.items():
    rows = completed_rows(path, EXPECTED["Seen"])
    if rows is None:
        rq4_final = False
        break
    rq4_raw[configuration] = rows

rq4_rows: list[dict[str, Any]] = []
for configuration in rq4_sources:
    rows = rq4_raw.get(configuration, [])
    if rows:
        metrics = summarize_rows(rows)
    else:
        metrics = {"N": np.nan, "PR": np.nan, "RR": np.nan, "IR": np.nan, "ATT": np.nan, "#Patch": np.nan}
    rq4_rows.append({"Configuration": configuration, **metrics})

rq4 = pd.DataFrame(rq4_rows)[["Configuration", "N", "PR", "RR", "IR", "ATT", "#Patch"]]
rq4_output = "rq4_sequential_escalation_final.csv" if rq4_final else "rq4_table_layout.csv"
rq4.to_csv(ANALYSIS_DIR / rq4_output, index=False)
display(rounded(rq4) if rq4_final else rq4.fillna("--"))

if rq4_final:
    sequential_rows = rq4_raw["Sequential"]
    rq4_test_rows = []
    for baseline_name in ("Progress", "Strict", "Answer"):
        baseline_rows = rq4_raw[baseline_name]
        rq4_test_rows.append({
            "Comparison": f"Sequential vs {baseline_name}",
            **exact_mcnemar(baseline_rows, sequential_rows),
            **{
                f"IR_{key}": value
                for key, value in exact_paired_binary(
                    baseline_rows,
                    sequential_rows,
                    key="improved",
                ).items()
            },
            **{
                f"PR_{key}": value
                for key, value in paired_bootstrap_mean_difference(
                    baseline_rows,
                    sequential_rows,
                    value_key="fixed_pass_rate",
                ).items()
            },
        })
    rq4_tests = add_holm_adjustment(pd.DataFrame(rq4_test_rows))
    rq4_tests = add_holm_adjustment(rq4_tests, column="IR_p_value")
    rq4_tests.to_csv(ANALYSIS_DIR / "rq4_sequential_paired_tests.csv", index=False)
    display(rounded(rq4_tests))

,Configuration,N,PR,RR,IR,ATT,#Patch
0,Progress,--,--,--,--,--,--
1,Strict,--,--,--,--,--,--
2,Answer,--,--,--,--,--,--
3,Sequential,--,--,--,--,--,--


## Completion manifest

This manifest prevents partial files from being presented as final results.

In [12]:
complete_methods = {
    setting: [method for method in METHOD_FILES[setting] if (setting, method) in rq1_raw]
    for setting in EXPECTED
}
manifest = {
    "RQ1": (
        "Complete"
        if len(complete_methods["Seen"]) == len(METHOD_FILES["Seen"])
        else f"Seen complete methods: {', '.join(complete_methods['Seen']) or 'none'}"
    ),
    "RQ2": "Complete" if rq2_ready else "Not complete",
    "RQ3": "Complete" if rq3_final else "Not complete",
    "RQ4": "Complete" if rq4_final else "Not complete",
    "RQ5": (
        "Complete"
        if len(complete_methods["Unseen"]) == len(METHOD_FILES["Unseen"])
        else f"Unseen complete methods: {', '.join(complete_methods['Unseen']) or 'none'}"
    ),
}
(ANALYSIS_DIR / "completion_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
display(pd.DataFrame(manifest.items(), columns=["RQ", "Available result"]))

,RQ,Available result
0,RQ1,Complete
1,RQ2,Not complete
2,RQ3,Not complete
3,RQ4,Not complete
4,RQ5,Unseen complete methods: none
